# Week 6 - Spark Architecture and Data Processing

## Objective

The objective of this assignment is to understand Spark architecture, execution flow, file formats, and DataFrame operations. The assignment demonstrates Spark transformations, filtering, schema handling, data processing pipelines, and performance concepts such as Lazy Evaluation, DAG, and Predicate Pushdown.

## Import Libraries

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

## Create Spark Session

SparkSession is the entry point for working with Spark DataFrames.

In [2]:
spark = SparkSession.builder \
    .appName("Week6_Spark_Assignment") \
    .getOrCreate()

print("Spark Session Created Successfully")

Spark Session Created Successfully


In [3]:
spark

## Load Datasets

The e-commerce datasets are loaded into Spark DataFrames for performing Spark transformations and architecture-related tasks.

In [4]:
customers_df = spark.read.csv(
    "../data/customers_dirty.csv",
    header=True,
    inferSchema=True
)

orders_df = spark.read.csv(
    "../data/orders_dirty.csv",
    header=True,
    inferSchema=True
)

products_df = spark.read.csv(
    "../data/products_dirty.csv",
    header=True,
    inferSchema=True
)

order_items_df = spark.read.csv(
    "../data/order_items_dirty.csv",
    header=True,
    inferSchema=True
)

## Explore Datasets

### Customers

In [5]:
customers_df.show(5)

+-----------+-------------+--------------------+-----------------+-------------+
|customer_id|customer_name|               email|registration_date|customer_type|
+-----------+-------------+--------------------+-----------------+-------------+
|  CUST00001|Aryan Maharaj|udantdewan@exampl...|       2023-12-03|          VIP|
|  CUST00002|  Pahal Balay|   invalid_email.com|       2025-08-20|      REGULAR|
|  CUST00003| Rushil Saini|saumyamall@exampl...|       2024-03-01|      REGULAR|
|  CUST00004|  Harini Mall|tanveernayar@exam...|       2023-11-18|          VIP|
|  CUST00005|Gunbir Parmer|aishani07@example...|       2024-01-06|      PREMIUM|
+-----------+-------------+--------------------+-----------------+-------------+
only showing top 5 rows


In [6]:
customers_df.printSchema()


root
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- registration_date: date (nullable = true)
 |-- customer_type: string (nullable = true)



### Orders

In [8]:
orders_df.show(5)

+---------+-----------+-------------------+---------+-----------+
| order_id|customer_id|         order_date|   status|region_code|
+---------+-----------+-------------------+---------+-----------+
|ORD000001|  CUST00961|2025-05-19 12:50:57|   PLACED|       EAST|
|ORD000002|  CUST00573|2023-07-13 10:52:55|CANCELLED|       EAST|
|ORD000003|  CUST00833|2024-06-07 00:41:07|  SHIPPED|    CENTRAL|
|ORD000004|  CUST00096|2023-07-29 15:18:24|CANCELLED|    CENTRAL|
|ORD000005|  CUST00371|2024-07-09 00:00:00|   PLACED|       EAST|
+---------+-----------+-------------------+---------+-----------+
only showing top 5 rows


In [9]:
orders_df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_date: timestamp (nullable = true)
 |-- status: string (nullable = true)
 |-- region_code: string (nullable = true)



### Products 

In [10]:
products_df.show(5)

+----------+---------------+--------+-----------+----------+
|product_id|   product_name|category|subcategory|cost_price|
+----------+---------------+--------+-----------+----------+
| PROD00001|Philips Outdoor|  Sports|    Outdoor|     39267|
| PROD00002|   Sony Fitness|  Sports|    Fitness|     34447|
| PROD00003|    LG Academic|   Books|   Academic|     29214|
| PROD00004|     Puma Decor|    Home|      Decor|     20213|
| PROD00005| Samsung Indoor|  Sports|     Indoor|     40152|
+----------+---------------+--------+-----------+----------+
only showing top 5 rows


In [11]:
products_df.printSchema()

root
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- subcategory: string (nullable = true)
 |-- cost_price: integer (nullable = true)



### Order_items

In [12]:
order_items_df.show(5)

+----------+---------+----------+--------+----------+----------------+
|   item_id| order_id|product_id|quantity|unit_price|discount_percent|
+----------+---------+----------+--------+----------+----------------+
|ITEM000001|ORD002900| PROD00135|       1|     13320|              25|
|ITEM000002|ORD004612| PROD00221|       5|     51413|               5|
|ITEM000003|ORD004368| PROD00093|       5|     34709|               0|
|ITEM000004|ORD004349| PROD00043|       5|     15401|              25|
|ITEM000005|ORD001357| PROD00163|       3|     10703|               0|
+----------+---------+----------+--------+----------+----------------+
only showing top 5 rows


In [13]:
order_items_df.printSchema()

root
 |-- item_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- discount_percent: integer (nullable = true)



In [14]:
print("Customers :", customers_df.count())
print("Orders :", orders_df.count())
print("Products :", products_df.count())
print("Order Items :", order_items_df.count())

Customers : 1010
Orders : 5000
Products : 250
Order Items : 15000


### Observation

All four datasets were successfully loaded into Spark DataFrames. The datasets represent customers, orders, products, and order items, forming a simple e-commerce data model. The inferred schemas allow further transformations, filtering, and analysis using Spark.

# Spark Architecture

## Q1. Explain the roles of the Driver, Cluster Manager, and Executor in a Spark application.

### Driver
The Driver is the main program of a Spark application. It creates the SparkSession, converts user code into execution plans, and coordinates the execution of tasks.

### Cluster Manager
The Cluster Manager is responsible for allocating resources to the Spark application. It manages the available computing resources and assigns them to different Spark jobs.

### Executor
Executors are worker processes that run tasks assigned by the Driver. They process data, perform computations, and return the results to the Driver.

### Spark Architecture Workflow

User Code

⬇

Driver Program

⬇

Cluster Manager

⬇

Executor 1 Executor 2 Executor 3

⬇

Process Data and Return Results

## Q13: In Spark Architecture, what is the difference between Client Mode and Cluster Mode? 

Where it runs: The main driver operates directly on personal computer or laptop.

Best used for: Writing code, testing, and actively developing a application.

Downside: Your application is entirely dependent on a machine. If our Wi-Fi drops, computer goes to sleep, or battery dies, the job stops immediately.

Advantage: It is incredibly easy to debug because the process is happening right in front of us on our local machine


Where it runs: The driver is deployed remotely and runs safely inside the cluster itself.

Best used for: Real-world, heavy-duty production environments where reliability is key.

The major downside: It can be slightly harder to debug since we aren't running it locally and have to check remote server logs to see what went wrong.

The biggest advantage: It is highly resilient. we can start a massive job, shut your laptop, And walk away the application will continue running on its own without needing your local connection.

### Observation

Spark applications consist of a Driver, Cluster Manager, and Executors working together to process data in a distributed environment. Understanding these components helped me to explain how Spark efficiently manages resources and executes tasks.

# Spark Execution

## Q2: How does Spark’s Lazy Evaluation strategy improve performance when chain-processing large datasets? 

While working on this assignment and by learning in the week I learned that Spark does not execute every transformation as soon as we write it. Instead it waits until an action like .show() or .count() is called. During this time Spark prepares an execution plan and combines multiple operations wherever possible. 
This reduces unnecessary processing and improves the overall performance, especially when working with large datasets.

## Q7: How does Spark use the Lineage Graph (DAG) to provide fault tolerance if a worker node fails? 

Spark keeps track of all the transformations that we perform on a DataFrame. This sequence of operations is called the Lineage Graph or DAG. If one of the worker nodes fails Spark does not need to reload everything from the beginning. Instead it uses the stored lineage to recompute only the missing data which makes the application more reliable.

## Q11: What is the difference between Transformations and Actions? Provide two examples of each. 

During this assignment and learning throughout the week, I noticed that not every Spark operation executes immediately.

- Transformations : create a new DataFrame but do not execute the computation right away.
- Actions :  trigger the execution and return the result.

Some common examples are shown below.

**Transformation Examples**
- filter()
- select()

**Action Examples**
- show()
- count()

In [15]:

# Transformation Examples

filtered_products = products_df.filter(col("category") == "Electronics")

selected_products = products_df.select("product_id", "product_name")

print("Transformation examples created successfully.")


Transformation examples created successfully.


In [16]:

# Action Examples

filtered_products.show(5)

print("Total Products:", filtered_products.count())

+----------+---------------+-----------+-----------+----------+
|product_id|   product_name|   category|subcategory|cost_price|
+----------+---------------+-----------+-----------+----------+
| PROD00008|      HP Mobile|Electronics|     Mobile|     11591|
| PROD00030|  Adidas Laptop|Electronics|     Laptop|     26961|
| PROD00038|Prestige Mobile|Electronics|     Mobile|     23909|
| PROD00040|    Nike Mobile|Electronics|     Mobile|     36193|
| PROD00052|    Sony Laptop|Electronics|     Laptop|     27393|
+----------+---------------+-----------+-----------+----------+
only showing top 5 rows
Total Products: 37


## Q15: When exploring a dataset, why is it safer to use .show(5) instead of .collect() on a multi-terabyte dataset?

While exploring a dataset, I found that using .show(5) is much safer because it only displays a few rows without loading the entire dataset into memory. On the other hand, .collect() brings all the data to the Driver machine. If the dataset is very large this can consume a lot of memory and may even crash the application. That's why using .show() is generally a better choice for exploring large datasets.

### Observation

In this section, I understood how Spark executes jobs internally. Learning about Lazy Evaluation, DAG, Transformations, and Actions helped me understand why Spark performs better than many traditional data processing systems when working with large datasets.

# File Formats and Data Processing

## Q3: Write a Spark command to read a CSV file located at "data/source.csv", ensuring the first row is treated as a header and inferSchema is enabled

For this assignment, I used Spark to load a CSV file by enabling the header option and schema inference. This helps Spark automatically detect column names and appropriate data types.

In [17]:
csv_df = spark.read.csv(
    "../data/products_dirty.csv",
    header=True,
    inferSchema=True
)

csv_df.show(5)

+----------+---------------+--------+-----------+----------+
|product_id|   product_name|category|subcategory|cost_price|
+----------+---------------+--------+-----------+----------+
| PROD00001|Philips Outdoor|  Sports|    Outdoor|     39267|
| PROD00002|   Sony Fitness|  Sports|    Fitness|     34447|
| PROD00003|    LG Academic|   Books|   Academic|     29214|
| PROD00004|     Puma Decor|    Home|      Decor|     20213|
| PROD00005| Samsung Indoor|  Sports|     Indoor|     40152|
+----------+---------------+--------+-----------+----------+
only showing top 5 rows


## Q4: What is the difference between CSV and Parquet in terms of storage (row-based vs. columnar) and why does it matter for performance? 

While learning about Spark I understood that CSV and Parquet store data differently.

- CSV is a row-based file format where each row is stored one after another. It is easy to read and share but usually takes more storage space.

- Parquet is a column-based file format. It stores values of the same column together, which allows Spark to read only the required columns instead of the whole dataset.

Because of this, Parquet generally provides faster query performance and better compression, especially when working with large datasets.

## Q9: Explain the concept of Predicate Pushdown in Parquet and how it affects the amount of data loaded into memory.

Predicate Pushdown is an optimization feature where Spark sends filter conditions directly to the storage layer. Instead of reading the complete dataset into memory, only the matching data is loaded.

This reduces disk I/O, improves performance and makes queries much faster when working with large datasets.

## Q12: Write the Spark command to load a Parquet file from "path/to/input", filter out any rows where user_id is null, and save the result as a CSV at "path/to/output". 

Since our datasets are CSV files, we'll first create a Parquet file and then answer the question properly.


For this question: I learned the standard Spark workflow of reading a Parquet file, applying a filter, and saving the processed data as a CSV. While implementing this on my local Windows system, I encountered a Hadoop configuration issue (`HADOOP_HOME` was not configured), which prevented the write operation from executing successfully. The following code represents the correct Spark implementation.

In [ ]:
# Standard Spark workflow

parquet_df = spark.read.parquet("path/to/input")

filtered_df = parquet_df.filter(
    col("user_id").isNotNull()
)

filtered_df.write.mode("overwrite").csv(
    "path/to/output",
    header=True
)

### Observation

This is the standard Spark workflow used to read a Parquet file, filter records, and save the processed data as a CSV file. During implementation on my local Windows environment, the write operation could not be executed because Hadoop (`HADOOP_HOME`) was not configured. In a properly configured Spark environment such as Databricks or a Linux cluster, the same code would execute successfully.

# DataFrame Operations

## Q5: Given a DataFrame df, write a query to select the columns product_id and price where the category is 'Electronics'. 

For this task, I filtered the products dataset to display only products from the Electronics category. Since my dataset contains a `cost_price` column instead of a generic `price` column, I used `cost_price` for this query.

In [22]:
electronics_products = (
    products_df
    .filter(col("category") == "Electronics")
    .select("product_id", "cost_price")
)

electronics_products.show()

+----------+----------+
|product_id|cost_price|
+----------+----------+
| PROD00008|     11591|
| PROD00030|     26961|
| PROD00038|     23909|
| PROD00040|     36193|
| PROD00052|     27393|
| PROD00061|     31196|
| PROD00065|     34668|
| PROD00073|     45052|
| PROD00078|      3877|
| PROD00088|     38707|
| PROD00092|     37681|
| PROD00104|     48195|
| PROD00111|      9424|
| PROD00115|     44294|
| PROD00123|      7810|
| PROD00124|     35428|
| PROD00130|     14438|
| PROD00132|     35882|
| PROD00135|     42254|
| PROD00141|      9682|
+----------+----------+
only showing top 20 rows


## Q6: Write the code to "revise" a DataFrame by renaming the column old_name to new_name and casting the price column from a String to a Double. 

Since my dataset contains cost_price instead of price, I first renamed the column to price and then converted its datatype to Double.

In [23]:
from pyspark.sql.types import DoubleType

products_updated = (
    products_df
    .withColumnRenamed("cost_price", "price")
    .withColumn(
        "price",
        col("price").cast(DoubleType())
    )
)

products_updated.printSchema()

products_updated.show(5)

root
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- subcategory: string (nullable = true)
 |-- price: double (nullable = true)

+----------+---------------+--------+-----------+-------+
|product_id|   product_name|category|subcategory|  price|
+----------+---------------+--------+-----------+-------+
| PROD00001|Philips Outdoor|  Sports|    Outdoor|39267.0|
| PROD00002|   Sony Fitness|  Sports|    Fitness|34447.0|
| PROD00003|    LG Academic|   Books|   Academic|29214.0|
| PROD00004|     Puma Decor|    Home|      Decor|20213.0|
| PROD00005| Samsung Indoor|  Sports|     Indoor|40152.0|
+----------+---------------+--------+-----------+-------+
only showing top 5 rows


## Q8: Write a query to filter a DataFrame df_orders for rows where the status is 'Completed' AND the amount is greater than 1000. 

The orders dataset does not contain an amount column, so I calculated it by multiplying quantity and unit_price after joining the orders and order items datasets.
The original question used the status "Completed". Since my dataset contains "SHIPPED" instead, I used that value to demonstrate the same filtering operation.

In [24]:
orders_amount = (
    orders_df
    .join(order_items_df, "order_id")
    .withColumn(
        "amount",
        col("quantity") * col("unit_price")
    )
)

In [27]:
completed_orders = (
    orders_amount
    .filter(
        (col("status") == "SHIPPED") &
        (col("amount") > 1000)
    )
)

completed_orders.show(5)

+---------+-----------+-------------------+-------+-----------+----------+----------+--------+----------+----------------+------+
| order_id|customer_id|         order_date| status|region_code|   item_id|product_id|quantity|unit_price|discount_percent|amount|
+---------+-----------+-------------------+-------+-----------+----------+----------+--------+----------+----------------+------+
|ORD003194|  CUST00287|2025-07-10 08:11:45|SHIPPED|       EAST|ITEM000008| PROD00231|       1|     26571|              10| 26571|
|ORD000003|  CUST00833|2024-06-07 00:41:07|SHIPPED|    CENTRAL|ITEM000010| PROD00020|       2|     47587|              10| 95174|
|ORD002871|       NULL|2023-08-07 17:43:47|SHIPPED|      SOUTH|ITEM000016| PROD00019|       2|     16590|              30| 33180|
|ORD003591|  CUST00784|2023-09-20 01:45:37|SHIPPED|       WEST|ITEM000021| PROD00077|       5|     31174|               5|155870|
|ORD002797|  CUST00689|2025-06-04 03:37:23|SHIPPED|       WEST|ITEM000023| PROD00225|     

## Q10: Write a code snippet to add a new column final_price which is the base_price multiplied by 1.18 (18% tax). 

In this task, I calculated the final price by adding 18% tax to the original product price.

In [26]:
products_final = (
    products_updated
    .withColumn(
        "final_price",
        col("price") * 1.18
    )
)

products_final.select(
    "product_id",
    "price",
    "final_price"
).show(5)

+----------+-------+-----------+
|product_id|  price|final_price|
+----------+-------+-----------+
| PROD00001|39267.0|   46335.06|
| PROD00002|34447.0|   40647.46|
| PROD00003|29214.0|   34472.52|
| PROD00004|20213.0|   23851.34|
| PROD00005|40152.0|   47379.36|
+----------+-------+-----------+
only showing top 5 rows


## Q14: Write a query to filter a dataset for rows where the region is 'North' OR the priority is 'High'. 

In [28]:
orders_priority = (
    orders_df
    .withColumn(
        "priority",
        when(col("status") == "SHIPPED", "High")
        .otherwise("Normal")
    )
)

In [29]:
filtered_orders = (
    orders_priority
    .filter(
        (col("region_code") == "NORTH") |
        (col("priority") == "High")
    )
)

filtered_orders.show(5)

+---------+-----------+-------------------+---------+-----------+--------+
| order_id|customer_id|         order_date|   status|region_code|priority|
+---------+-----------+-------------------+---------+-----------+--------+
|ORD000003|  CUST00833|2024-06-07 00:41:07|  SHIPPED|    CENTRAL|    High|
|ORD000006|  CUST00317|2026-05-21 02:41:52|  SHIPPED|      SOUTH|    High|
|ORD000008|  CUST00232|2024-04-21 09:51:56|  SHIPPED|      SOUTH|    High|
|ORD000009|  CUST00080|2023-11-15 15:59:15|DELIVERED|      NORTH|  Normal|
|ORD000013|  CUST00158|2024-06-06 10:46:33|  SHIPPED|      SOUTH|    High|
+---------+-----------+-------------------+---------+-----------+--------+
only showing top 5 rows


# Observations

During this assignment I explored different aspects of Apache Spark beyond basic DataFrame operations. I learned how Spark processes data using the Driver, Cluster Manager, and Executors, and how features like Lazy Evaluation and DAG help improve performance.
 I also worked with filtering, schema modifications, data type conversions, and basic file handling using CSV and Parquet. 
 Overall, this assignment gave me a better understanding of how Spark is used in real-world data engineering workflows.

# Conclusion

This assignment helped me strengthen my understanding of Apache Spark fundamentals. Along with learning the architecture, I also practiced common DataFrame operations, schema handling, and filtering techniques using PySpark. Overall, this assignment improved my confidence in working with Spark and gave me practical exposure to concepts that are commonly used in data engineering projects.